In [1]:
import warnings
from rdkit import RDLogger

# 屏蔽 RDKit 警告
RDLogger.DisableLog('rdApp.*')

# 或屏蔽所有 Python 警告
warnings.filterwarnings("ignore")
# 屏蔽 LightGBM 警告
warnings.filterwarnings("ignore", category=UserWarning, module="lightgbm")

In [5]:
import torch
from sklearn.model_selection import StratifiedKFold
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics import precision_recall_curve, auc
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib
import optuna
from rdkit.Chem import Descriptors, AllChem
from tqdm import tqdm  # 导入tqdm
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold





In [2]:
# 函数：将SMILES转换为分子描述符和指纹
def smiles_to_features(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # 提取描述符
    descriptors = [
        Descriptors.MolWt(mol),  # 分子量
        Descriptors.MolLogP(mol),  # LogP
        Descriptors.NumHDonors(mol),  # 氢键供体数量
        Descriptors.NumHAcceptors(mol)  # 氢键受体数量
    ]
    # 生成Morgan指纹
    fingerprint = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fingerprint_array = np.zeros((2048,))
    Chem.DataStructs.ConvertToNumpyArray(fingerprint, fingerprint_array)
    # 合并描述符和指纹
    features = np.concatenate([descriptors, fingerprint_array])
    return features


In [3]:
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from tqdm import tqdm
import optuna
import numpy as np

def train_evaluate_regression_model_with_optuna(model_name, model_class, param_func, X, y, groups):
    def objective(trial):
        params = param_func(trial)
        model = model_class(**params)

        gkf = GroupKFold(n_splits=10)
        maes = []

        for train_idx, val_idx in tqdm(gkf.split(X, y, groups=groups), total=10, desc=f"Training {model_name}"):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)

            # ✅ 计算 MAE
            mae = mean_absolute_error(y_val, y_pred)
            maes.append(mae)

        return np.mean(maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)

    print(f'Best parameters for {model_name}: {study.best_params}')
    print(f'Best mean MAE: {study.best_value:.4f}')
    

In [6]:
# 数据预处理
df = pd.read_excel('../algae_EC10_unique.xlsx')
labels = df['mgperL'].values
smiles_list = df['SMILES_Canonical_RDKit'].tolist()
endpoints_a = df['endpoint']
Duration_Values_a = df['Duration_Value'].values
effects_a = df['effect']


In [7]:

features = []
new_labels = []
new_smiles_list = []
endpoints = []
Duration_Values = []
effects =[]


for smiles, label,a,b,c in zip(smiles_list, labels,Duration_Values_a,effects_a,endpoints_a):
    feature = smiles_to_features(smiles)
    if feature is not None:
        features.append(feature)
        new_labels.append(label)
        new_smiles_list.append(smiles)
        Duration_Values.append(a)
        effects.append(b)
        endpoints.append(c)

X = np.array(features)
y = np.array(new_labels)
groups = new_smiles_list  # 可直接用于 GroupKFold




In [8]:
def encode_column(zz):
    zz_series = pd.Series(zz)  # 转换为 Series
    unique_values = zz_series.unique()
    if len(unique_values) > 1:
        encoder = OneHotEncoder(sparse_output=False)
        return encoder.fit_transform(zz_series.values.reshape(-1, 1))
    else:
        return None  # 只有一种类别时忽略

Duration_Values =pd.Series(Duration_Values)


# 编码 effect、endpoint 和 species_group 列
effect_encoded = encode_column(effects)
endpoint_encoded = encode_column(endpoints)
#species_encoded = encode_column(df, 'species_group')

# # 将需要的列拼接成输入 X
X = np.hstack((X, Duration_Values.values.reshape(-1, 1)))

# # 拼接编码后的列（如果存在）
for encoded_feature in [effect_encoded, endpoint_encoded]:
     if encoded_feature is not None:
         X = np.hstack((X, encoded_feature))



y=np.log1p(y)

In [9]:
def xgb_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),   # L1 正则
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0)  # L2 正则
    }
from xgboost import XGBRegressor

train_evaluate_regression_model_with_optuna(
    "XGBoost",
    XGBRegressor,
    xgb_param_func,
    X, y, groups
)

[I 2025-05-16 09:47:29,694] A new study created in memory with name: no-name-3140fdeb-69ac-4821-83af-7f59af43bdeb
Training XGBoost: 100%|██████████| 10/10 [00:41<00:00,  4.13s/it]
[I 2025-05-16 09:48:11,052] Trial 0 finished with value: 1.127303009089911 and parameters: {'n_estimators': 333, 'max_depth': 10, 'learning_rate': 0.06722881802249443, 'subsample': 0.800162448572537, 'colsample_bytree': 0.6198686114139847, 'reg_alpha': 0.9339038479113909, 'reg_lambda': 0.8430225255532279}. Best is trial 0 with value: 1.127303009089911.
Training XGBoost: 100%|██████████| 10/10 [01:11<00:00,  7.15s/it]
[I 2025-05-16 09:49:22,565] Trial 1 finished with value: 1.115736912017997 and parameters: {'n_estimators': 419, 'max_depth': 16, 'learning_rate': 0.0599667068280911, 'subsample': 0.9175273889274178, 'colsample_bytree': 0.6509887620090443, 'reg_alpha': 0.12474249774044988, 'reg_lambda': 0.6975174097649423}. Best is trial 1 with value: 1.115736912017997.
Training XGBoost: 100%|██████████| 10/10 [0

Best parameters for XGBoost: {'n_estimators': 547, 'max_depth': 16, 'learning_rate': 0.026418787686066283, 'subsample': 0.6188013405471477, 'colsample_bytree': 0.9970208205606014, 'reg_alpha': 0.6389473970031672, 'reg_lambda': 0.3632950015284354}
Best mean MAE: 1.0978


In [10]:
from lightgbm import LGBMRegressor

def lgbm_param_func(trial):
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 1.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 1.0),
        'verbose': -1
    }

print("Training LightGBM (Poisson)...")
train_evaluate_regression_model_with_optuna(
    "LightGBM",
    lambda **params: LGBMRegressor(objective="poisson", **params),  # ✅ 加入 Poisson 目标
    lgbm_param_func,
    X, y, groups
)


[I 2025-05-16 10:35:30,333] A new study created in memory with name: no-name-5ce14a7e-eeac-4217-9332-4f96d816ea24


Training LightGBM (Poisson)...


Training LightGBM: 100%|██████████| 10/10 [00:14<00:00,  1.41s/it]
[I 2025-05-16 10:35:44,471] Trial 0 finished with value: 1.127330545704504 and parameters: {'n_estimators': 415, 'max_depth': 18, 'num_leaves': 38, 'learning_rate': 0.03767519058936203, 'feature_fraction': 0.9774001932081017, 'bagging_fraction': 0.8442956210698775, 'bagging_freq': 5, 'reg_alpha': 0.6254689474139133, 'reg_lambda': 0.04048746061972497}. Best is trial 0 with value: 1.127330545704504.
Training LightGBM: 100%|██████████| 10/10 [00:08<00:00,  1.24it/s]
[I 2025-05-16 10:35:52,554] Trial 1 finished with value: 1.247057514087824 and parameters: {'n_estimators': 136, 'max_depth': 16, 'num_leaves': 108, 'learning_rate': 0.019890604330757202, 'feature_fraction': 0.9055152910421251, 'bagging_fraction': 0.7187000145925143, 'bagging_freq': 1, 'reg_alpha': 0.7674481356340422, 'reg_lambda': 0.9915709953178917}. Best is trial 0 with value: 1.127330545704504.
Training LightGBM: 100%|██████████| 10/10 [00:05<00:00,  1.99it

Best parameters for LightGBM: {'n_estimators': 539, 'max_depth': 20, 'num_leaves': 250, 'learning_rate': 0.08748337263849608, 'feature_fraction': 0.8061814340487374, 'bagging_fraction': 0.8982902904679516, 'bagging_freq': 4, 'reg_alpha': 0.5833748683869527, 'reg_lambda': 0.7680801374089827}
Best mean MAE: 1.0856


In [11]:
import random

# 固定随机种子
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # for multi-GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)  # 设置固定种子

In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import optuna
import numpy as np


class DNNWithSoftplus(nn.Module):
    def __init__(self, input_dim, hidden_sizes, activation):
        super().__init__()
        act_fn = {
            'relu': nn.ReLU(),
            'logistic': nn.Sigmoid(),
            'tanh': nn.Tanh()
        }[activation]
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev_dim, h), act_fn]
            prev_dim = h
        layers += [nn.Linear(prev_dim, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return F.softplus(self.net(x)).squeeze(-1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_dnn_with_optuna_pytorch(X, y, groups, device=device):
    def dnn_param_func(trial):
        return {
            'hidden_layer_sizes': trial.suggest_categorical(
                'hidden_layer_sizes', [(50,), (100,), (150,), (100, 50), (150, 100, 50)]
            ),
            'activation': trial.suggest_categorical('activation', ['relu', 'logistic', 'tanh']),
            'alpha': trial.suggest_float('alpha', 1e-5, 1e-2, log=True),
            'learning_rate': trial.suggest_float('learning_rate_init', 1e-4, 1e-2, log=True),
            'optimizer': trial.suggest_categorical('solver', ['adam', 'sgd'])
        }

    def objective(trial):
        params = dnn_param_func(trial)
        model = DNNWithSoftplus(
            input_dim=X.shape[1],
            hidden_sizes=params['hidden_layer_sizes'],
            activation=params['activation']
        ).to(device)

        optimizer = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD
        }[params['optimizer']](model.parameters(), lr=params['learning_rate'], weight_decay=params['alpha'])

        loss_fn = nn.MSELoss()
        gkf = GroupKFold(n_splits=10)
        fold_maes = []

        for train_idx, val_idx in gkf.split(X, y, groups=groups):
            X_train, y_train = X[train_idx], y[train_idx]
            X_val, y_val = X[val_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
            train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)

            model.train()
            for epoch in range(100):
                for xb, yb in train_loader:
                    xb, yb = xb.to(device), yb.to(device)
                    optimizer.zero_grad()
                    pred = model(xb)
                    loss = loss_fn(pred, yb)
                    loss.backward()
                    optimizer.step()

            model.eval()
            with torch.no_grad():
                val_preds = model(torch.tensor(X_val).float().to(device)).cpu().numpy()
                mae = mean_absolute_error(y_val, val_preds)
                fold_maes.append(mae)

        return np.mean(fold_maes)

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30)
    print("\n✅ Best Parameters Found:")
    print(study.best_params)
    print(f"Mean MAE = {study.best_value:.4f}")
    return study.best_params


best_dnn_params = train_dnn_with_optuna_pytorch(X, y, groups)

[I 2025-05-16 12:55:38,684] A new study created in memory with name: no-name-b6041dea-c4ef-4cca-a854-48cc5774e2ab
[I 2025-05-16 12:56:41,530] Trial 0 finished with value: 1.2052129904168876 and parameters: {'hidden_layer_sizes': (50,), 'activation': 'logistic', 'alpha': 0.0022959177053963314, 'learning_rate_init': 0.00026423770635394323, 'solver': 'sgd'}. Best is trial 0 with value: 1.2052129904168876.
[I 2025-05-16 12:57:47,589] Trial 1 finished with value: 0.6499931513409078 and parameters: {'hidden_layer_sizes': (50,), 'activation': 'relu', 'alpha': 3.4510761019851425e-05, 'learning_rate_init': 0.0009654960052458606, 'solver': 'sgd'}. Best is trial 1 with value: 0.6499931513409078.
[I 2025-05-16 12:59:07,330] Trial 2 finished with value: 0.5721374558707348 and parameters: {'hidden_layer_sizes': (50,), 'activation': 'relu', 'alpha': 5.9695201442772074e-05, 'learning_rate_init': 0.0010697712743054455, 'solver': 'adam'}. Best is trial 2 with value: 0.5721374558707348.
[I 2025-05-16 13:


✅ Best Parameters Found:
{'hidden_layer_sizes': (100,), 'activation': 'tanh', 'alpha': 7.986958425859787e-05, 'learning_rate_init': 0.0005677745722341227, 'solver': 'adam'}
Mean MAE = 0.5112
